# Lesson 02 — Fine-Tuning & PEFT (LoRA)

Covers: full fine-tuning vs. PEFT, LoRA intuition & implementation, Hugging Face `transformers` + `peft` workflow.

**Interview relevance:** Any senior ML role touching LLMs will ask about fine-tuning trade-offs.

## 1. Why Full Fine-Tuning is Expensive

Fine-tuning GPT-2 (117M params) requires ~1.5 GB GPU RAM just for weights (fp32). With optimizer states (Adam stores 2× copies) that's ~4.5 GB — before activations. For 7B-param models this is infeasible on single GPUs.

**Key trade-offs:**
| Method | Trainable Params | Memory | Catastrophic Forgetting |
|--------|-----------------|--------|-------------------------|
| Full FT | 100% | Very High | Risk |
| Adapter | ~1-3% | Low | Low |
| LoRA | ~0.1-1% | Very Low | Very Low |
| Prompt Tuning | <0.01% | Minimal | None |

## 2. LoRA — Low-Rank Adaptation (from scratch)

In [ ]:
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    Wraps an existing Linear layer with a low-rank update:
        W' = W + alpha/r * (B @ A)
    W is frozen; only A and B are trained.
    """
    def __init__(self, linear: nn.Linear, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        self.linear = linear
        self.rank = rank
        self.alpha = alpha
        in_f, out_f = linear.in_features, linear.out_features

        # Freeze original weights
        for p in self.linear.parameters():
            p.requires_grad = False

        # LoRA matrices — A init: kaiming, B init: zeros (so delta=0 at start)
        self.A = nn.Parameter(torch.empty(rank, in_f))
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        base = self.linear(x)
        lora = (x @ self.A.T @ self.B.T) * (self.alpha / self.rank)
        return base + lora

# Example: replace a projection layer in a small transformer
base_linear = nn.Linear(512, 512)
lora_linear = LoRALinear(base_linear, rank=8, alpha=16)

x = torch.randn(2, 10, 512)
out = lora_linear(x)
print(f"Output shape: {out.shape}")

total = sum(p.numel() for p in lora_linear.parameters())
trainable = sum(p.numel() for p in lora_linear.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,} ({100*trainable/total:.1f}%)")


## 3. Applying LoRA to All Attention Projections

In [ ]:
def apply_lora_to_model(model, rank=8, alpha=16, target_modules=("q_proj", "v_proj")):
    """Replace target linear layers with LoRA-wrapped versions in-place."""
    for name, module in model.named_modules():
        for attr in target_modules:
            if hasattr(module, attr):
                original = getattr(module, attr)
                if isinstance(original, nn.Linear):
                    setattr(module, attr, LoRALinear(original, rank=rank, alpha=alpha))
    return model

# Demo with a toy transformer
class ToyAttention(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.out_proj = nn.Linear(d, d)

    def forward(self, x):
        q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        attn = torch.softmax(q @ k.transpose(-2,-1) / (128**0.5), dim=-1)
        return self.out_proj(attn @ v)

toy = ToyAttention()
toy = apply_lora_to_model(toy, rank=4)

trainable = sum(p.numel() for p in toy.parameters() if p.requires_grad)
total     = sum(p.numel() for p in toy.parameters())
print(f"After LoRA — Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")


## 4. Hugging Face PEFT Workflow

In [ ]:
# pip install transformers peft datasets accelerate

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# LoRA config — target the attention Q and V projections
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],   # DistilBERT attention layer names
    bias="none",
)

peft_model = get_peft_model(base_model, peft_config)
peft_model.print_trainable_parameters()
# Expected: ~0.5% trainable parameters


## 5. When to Choose Which Method

| Scenario | Recommended |
|---|---|
| Large LLM (≥7B), limited GPU | LoRA / QLoRA |
| Domain adaptation (same task) | Full fine-tune last N layers |
| New task, small dataset (<1k) | Prompt tuning or few-shot |
| Catastrophic forgetting is a concern | LoRA (frozen base) |
| Need interpretable representations | Adapter layers |

**Q: Why does LoRA init B=0?**  
So the LoRA delta is zero at the start of training, making the model identical to the pretrained base. Training starts from the pretrained optimum rather than a random perturbation.

**Q: What is rank r and how do you choose it?**  
Rank controls the expressivity of the update. r=4 for minimal footprint, r=16–64 for complex task shifts. Too high ≈ full fine-tuning cost; too low underfits. Sweep with small validation set.